# Walkthrough — L3: Regularization Strategies & Cross-Validation

**O que este walkthrough faz:** explica célula por célula o notebook `L3_Regularization_CrossValidation.ipynb`, com foco nas ferramentas Python/sklearn/scipy/Matplotlib usadas, por que cada escolha foi feita e como replicar os padrões.

**Conceitos matemáticos** já são explicados no notebook original — aqui o foco é no **código**.

---

## Roteiro
1. Setup — `warnings.filterwarnings` e `matplotlib.patches`
2. Visualização geométrica — `ax.contour`, `ax.fill`, `plt.Circle`
3. `ElasticNet` — parâmetros e Regularization Paths
4. Gradient Descent manual — Early Stopping implementado do zero
5. Dropout — `np.random.binomial` e `mpatches.Patch`
6. Label Smoothing — barras com valores anotados
7. `scipy.stats.norm` e `scipy.stats.laplace` — objetos de distribuição
8. Cross-Validation variants — `KFold`, `StratifiedKFold`, `LeaveOneOut`
9. Boxplots com `ax.boxplot` e `patch_artist`
10. Coarse-to-Fine Log-Grid

---
## 1. Setup — `warnings` e `matplotlib.patches`

### O que o código faz

```python
import matplotlib.patches as mpatches
from matplotlib.colors import LogNorm
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.model_selection import (
    KFold, StratifiedKFold, LeaveOneOut,
    cross_val_score, train_test_split, GridSearchCV
)
import warnings
warnings.filterwarnings('ignore')
```

### `matplotlib.patches` — formas geométricas

`mpatches` contém classes para desenhar **formas geométricas** em gráficos:

| Classe | O que desenha |
|--------|---------------|
| `mpatches.Circle((x,y), r)` | Círculo centrado em (x,y) com raio r |
| `mpatches.Patch(color, label)` | Retângulo colorido para legenda customizada |
| `mpatches.FancyBboxPatch(xy, w, h, boxstyle)` | Retângulo com bordas arredondadas |
| `mpatches.FancyArrowPatch(...)` | Seta customizada |

Patches são adicionados ao eixo com `ax.add_patch(patch)` — diferente de `ax.plot` que desenha linhas/pontos.

### `LogNorm` — escala logarítmica em colormaps

```python
from matplotlib.colors import LogNorm
ax.contourf(X, Y, Z, norm=LogNorm())
```

Quando `Z` tem valores que variam em várias ordens de magnitude, usar `LogNorm()` distribui as cores uniformemente em escala log. (No notebook L3, importada mas usada marginalmente.)

---
## 2. Visualização Geométrica — `contour`, `ax.fill` e `plt.Circle`

### O que o código faz

```python
T1, T2 = np.meshgrid(np.linspace(-3, 3, 300), np.linspace(-3, 3, 300))
Loss = (T1 - 2)**2 + 0.5*(T2 - 1.5)**2

# Hard L2: círculo
circulo = plt.Circle((0, 0), 1.5, fill=False, color='navy', lw=2.5, ls='--')
ax.add_patch(circulo)
ax.fill(1.5*np.cos(theta_plot), 1.5*np.sin(theta_plot), alpha=0.12, color='navy')

# L1: diamante
diamond_x = [R1, 0, -R1, 0, R1]
diamond_y = [0, R1,  0, -R1, 0]
ax.fill(diamond_x, diamond_y, alpha=0.15, color='tab:orange')
ax.plot(diamond_x, diamond_y, color='tab:orange', lw=2.5)
```

### `np.meshgrid` — grid 2D para contornos

```python
T1, T2 = np.meshgrid(np.linspace(-3, 3, 300), np.linspace(-3, 3, 300))
Loss = (T1 - 2)**2 + 0.5*(T2 - 1.5)**2
```

`meshgrid` cria duas grades 2D de shape `(300, 300)` onde:
- `T1[i, j]` é o valor de θ₁ no ponto (i, j) do grid
- `T2[i, j]` é o valor de θ₂ no ponto (i, j) do grid

A expressão `(T1 - 2)**2 + 0.5*(T2 - 1.5)**2` avalia a função de loss em todos os 300×300 = 90.000 pontos de uma vez (vetorizado).

### `ax.contour` vs `ax.contourf`

```python
ax.contour(T1, T2, Loss, levels=15, cmap='RdYlGn_r', alpha=0.7)    # linhas
ax.contourf(T1, T2, Loss, levels=15, cmap='RdYlGn_r', alpha=0.7)   # regiões preenchidas
```

- `contour` → apenas as linhas de nível (isotermas)
- `contourf` → regiões preenchidas entre linhas (mais visual, usado em L3)
- `levels=15` → quantas curvas de nível
- `cmap='RdYlGn_r'` → colormap vermelho-amarelo-verde revertido (`_r`): vermelho = alto (pior), verde = baixo (melhor)

### `plt.Circle` vs `ax.fill` para o círculo L2

```python
# Método 1: usando plt.Circle (patch geométrico)
circulo = plt.Circle((0, 0), 1.5, fill=False, color='navy', lw=2.5)
ax.add_patch(circulo)

# Método 2: preenchimento manual com parametrização trigonométrica
theta_plot = np.linspace(0, 2*np.pi, 300)
ax.fill(1.5*np.cos(theta_plot), 1.5*np.sin(theta_plot), alpha=0.12, color='navy')
```

O notebook usa os dois juntos: `Circle` para o contorno exato, `fill` para o preenchimento transparente. `ax.fill(x_array, y_array)` preenche o polígono definido pelos pontos — mais flexível que `Circle` para formas irregulares.

### O Diamante L1 — 5 pontos

```python
diamond_x = [R1, 0, -R1, 0, R1]  # 4 vértices + fechamento
diamond_y = [0, R1,  0, -R1, 0]
ax.fill(diamond_x, diamond_y, alpha=0.15, color='tab:orange')
```

Um diamante em 2D tem 4 vértices: (R,0), (0,R), (-R,0), (0,-R). O 5º ponto repete o 1º para fechar o polígono. `ax.fill` interpola entre os pontos e preenche o interior.

### `ax.set_aspect('equal')` — escala igual nos eixos

```python
ax.set_aspect('equal')
```

Garante que 1 unidade no eixo x ocupa o mesmo espaço visual que 1 unidade no eixo y. **Essencial** para gráficos geométricos (círculos, diamantes) — sem isso, um círculo pareceria elipse.

---
## 3. `ElasticNet` e Regularization Paths

### O que o código faz

```python
lambdas_path = np.logspace(-4, 2, 80)

coefs_ridge = [Ridge(alpha=lam).fit(X_sp, y_sp).coef_ for lam in lambdas_path]
coefs_lasso = [Lasso(alpha=lam, max_iter=10000).fit(X_sp, y_sp).coef_ for lam in lambdas_path]
coefs_enet  = [ElasticNet(alpha=lam, l1_ratio=0.5, max_iter=10000).fit(X_sp, y_sp).coef_
               for lam in lambdas_path]

coefs_ridge = np.array(coefs_ridge)
coefs_lasso = np.array(coefs_lasso)
```

### List comprehension com `.fit().coef_`

```python
[Ridge(alpha=lam).fit(X_sp, y_sp).coef_ for lam in lambdas_path]
```

Cadeia de métodos: cria o objeto → treina → extrai coeficientes. Funciona porque `.fit()` retorna o próprio estimador (`self`).

Resultado: lista de 80 arrays de coeficientes. `np.array(...)` converte para shape `(80, 20)` — 80 lambdas × 20 features.

### Parâmetros de `ElasticNet`

```python
ElasticNet(alpha=lam, l1_ratio=0.5, max_iter=10000)
```

- `alpha` = força total de regularização λ (equivale ao `alpha` de Ridge e Lasso)
- `l1_ratio` = proporção de L1 vs L2:
  - `l1_ratio=0` → equivale a Ridge puro
  - `l1_ratio=1` → equivale a Lasso puro
  - `l1_ratio=0.5` → mistura igual
- `max_iter=10000` → número máximo de iterações do solver; aumentar evita `ConvergenceWarning`

### Plotando Regularization Paths

```python
for j in range(20):
    alpha_j = 0.9 if np.max(np.abs(coefs[:, j])) > 5 else 0.4
    ax.plot(lambdas_path, coefs[:, j], lw=1.5, alpha=alpha_j)
ax.invert_xaxis()  # lendo da direita (λ grande) para esquerda (λ pequeno)
```

`coefs[:, j]` — todos os valores de lambda para o j-ésimo coeficiente (coluna `j` do array 2D). `alpha_j` variável destaca os coeficientes importantes (magnitude > 5) com mais opacidade.

`ax.invert_xaxis()` inverte para o gráfico ler da **direita para esquerda** conforme λ aumenta — convenção da literatura para Regularization Paths.

---
## 4. Gradient Descent Manual — Early Stopping Implementado do Zero

### O que o código faz

```python
theta = np.zeros(grau + 1)  # inicialização em zeros
eta = 1e-3                   # learning rate
patience = 200

historico_treino, historico_val = [], []
melhor_val, melhor_theta, melhor_epoca = np.inf, theta.copy(), 0
sem_melhora = 0
epoca_stop = n_epocas

for ep in range(n_epocas):
    grad = -2/len(x_tr) * B_tr.T @ (y_tr - B_tr @ theta)
    theta = theta - eta * grad
    mse_t = mean_squared_error(y_tr, B_tr @ theta)
    mse_v = mean_squared_error(y_val, B_val @ theta)
    historico_treino.append(mse_t)
    historico_val.append(mse_v)
    if mse_v < melhor_val:
        melhor_val = mse_v
        melhor_theta = theta.copy()  # salva cópia dos melhores pesos
        melhor_epoca = ep
        sem_melhora = 0
    else:
        sem_melhora += 1
        if sem_melhora >= patience and epoca_stop == n_epocas:
            epoca_stop = ep
```

### O gradiente da MSE

```python
grad = -2/len(x_tr) * B_tr.T @ (y_tr - B_tr @ theta)
```

Esta é a derivada da MSE em relação a θ:
- `B_tr @ theta` = previsões `ŷ` (shape: `(N,)`)
- `y_tr - B_tr @ theta` = resíduos `(y - ŷ)` (shape: `(N,)`)
- `B_tr.T @ (y - ŷ)` = produto matricial → shape: `(M,)` onde M é o número de basis functions
- `-2/N` = fator da derivada do MSE

**Update:** `theta = theta - eta * grad` — subtrai porque gradiente aponta para a direção de **maior** crescimento; queremos o **menor** MSE.

### Early Stopping — lógica do contador `patience`

```python
melhor_val = np.inf      # inicializa como infinito (qualquer valor é melhor)
melhor_theta = theta.copy()  # .copy() é CRÍTICO
sem_melhora = 0

if mse_v < melhor_val:
    melhor_val = mse_v
    melhor_theta = theta.copy()  # checkpoint dos melhores pesos
    sem_melhora = 0
else:
    sem_melhora += 1
    if sem_melhora >= patience:
        epoca_stop = ep
```

**Por que `theta.copy()`?** `melhor_theta = theta` criaria uma **referência** ao mesmo array — quando `theta` mudasse na próxima iteração, `melhor_theta` também mudaria. `.copy()` cria um array independente — um snapshot dos pesos naquele momento.

**`np.inf`** é o infinito positivo do NumPy — qualquer número real é menor que `np.inf`. Inicializar `melhor_val = np.inf` garante que o primeiro valor de validação sempre seja aceito como "melhor".

### Visualizando com `ax.semilogy`

```python
ax.semilogy(historico_treino, 'tab:green', lw=2, label='Loss Treino')
ax.semilogy(historico_val,   'tab:red',   lw=2, label='Loss Validação')
ax.axvline(melhor_epoca, color='blue', ls='--', lw=2)
ax.axvline(epoca_stop, color='purple', ls=':', lw=2)
```

`semilogy` = eixo y em escala logarítmica. Essencial quando a loss cai várias ordens de magnitude — em escala linear, as últimas épocas parecem planas mesmo que ainda haja progresso significativo.

---
## 5. Dropout — `np.random.binomial` e `mpatches.Patch`

### O que o código faz

```python
h = np.random.randn(20)  # ativações de 20 neurônios

for ax, p_drop, titulo in [(axes[0], 0.0, ...), (axes[1], 0.5, ...), (axes[2], 0.8, ...)]:
    mask = np.random.binomial(1, 1 - p_drop, size=len(h))
    h_dropped = h * mask
    cores_bar = ['tab:blue' if m == 1 else 'tab:red' for m in mask]
    ax.bar(range(len(h)), h_dropped, color=cores_bar, alpha=0.8)
```

### `np.random.binomial` — amostras Bernoulli

```python
mask = np.random.binomial(1, 1 - p_drop, size=len(h))
```

`binomial(n, p, size)` gera `size` amostras da distribuição Binomial(n, p).

Com `n=1`, Binomial(1, p) = **Bernoulli(p)**: cada elemento é 0 com probabilidade `p_drop` ou 1 com probabilidade `1 - p_drop`.

- `p_drop = 0.5` → `mask` tem ~50% de zeros e ~50% de uns
- `h * mask` → zeros nos neurônios "dropados"

**Alternativa mais moderna** (PyTorch):
```python
mask = torch.bernoulli(torch.full_like(h, 1 - p_drop))
```

### `ax.bar` com cores por elemento

```python
cores_bar = ['tab:blue' if m == 1 else 'tab:red' for m in mask]
ax.bar(range(len(h)), h_dropped, color=cores_bar, alpha=0.8)
```

`ax.bar` aceita uma **lista de cores** do mesmo tamanho que os dados — cada barra pode ter cor diferente. O list comprehension atribui azul aos neurônios ativos e vermelho aos dropados.

### `mpatches.Patch` — legenda customizada

```python
azul_patch = mpatches.Patch(color='tab:blue', label='Neurônio ativo')
verm_patch = mpatches.Patch(color='tab:red',  label='Neurônio zerado (dropped)')
axes[2].legend(handles=[azul_patch, verm_patch], fontsize=9)
```

Quando as cores do plot não são automaticamente capturadas pela legenda, cria-se `Patch` manualmente — são retângulos coloridos que representam categorias. Passados via `handles=[...]` para `legend()`.

**Por que em `axes[2]` apenas?** Evitar legenda duplicada em cada subplot — colocar apenas no último é suficiente para o leitor entender.

---
## 6. Label Smoothing — Barras com Valores Anotados

### O que o código faz

```python
y_hard   = np.array([1, 0, 0, 0, 0])
y_smooth = y_hard * (1 - eps) + eps / K  # fórmula de smoothing

bars = ax.bar(range(K), y_smooth, color=cores, alpha=0.85, edgecolor='black')
for bar, v in zip(bars, y_smooth):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f'{v:.3f}',
            ha='center', fontsize=9, fontweight='bold')
```

### A fórmula de Label Smoothing vetorizada

```python
y_smooth = y_hard * (1 - eps) + eps / K
```

Com `y_hard = [1, 0, 0, 0, 0]`, `eps = 0.1`, `K = 5`:
- Para a classe correta (índice 0): `1 * 0.9 + 0.1/5 = 0.9 + 0.02 = 0.92`
- Para as outras: `0 * 0.9 + 0.1/5 = 0.02`

O broadcasting faz `eps / K` (escalar) ser somado a cada elemento do array.

### Anotando valores nas barras

```python
for bar, v in zip(bars, y_smooth):
    ax.text(
        bar.get_x() + bar.get_width()/2,  # centro horizontal da barra
        bar.get_height() + 0.01,           # logo acima do topo
        f'{v:.3f}',                        # texto formatado
        ha='center'                        # alinhado ao centro
    )
```

`ax.bar()` retorna um objeto `BarContainer` — uma coleção de `Rectangle` patches. Iterar com `zip(bars, y_smooth)` dá acesso a cada par `(patch_da_barra, valor)`. Métodos do patch:
- `bar.get_x()` — borda esquerda da barra
- `bar.get_width()` — largura da barra
- `bar.get_height()` — altura (= o valor)

---
## 7. `scipy.stats` — Objetos de Distribuição

### O que o código faz

```python
from scipy.stats import norm, laplace

theta_vis = np.linspace(-4, 4, 400)

for tau, lw_val, alpha_val in [(0.5, 3, 1.0), (1.0, 2, 0.7), (2.0, 1.5, 0.5)]:
    ax.plot(theta_vis, norm.pdf(theta_vis, 0, tau), lw=lw_val, alpha=alpha_val,
            label=f'$\\mathcal{{N}}(0, {tau}^2)$')

for b, lw_val, alpha_val in [(0.5, 3, 1.0), (1.0, 2, 0.7), (2.0, 1.5, 0.5)]:
    ax.plot(theta_vis, laplace.pdf(theta_vis, 0, b), ...)
```

### Objetos de distribuição do scipy.stats

`norm` e `laplace` são objetos de distribuição — não funções simples. Eles têm vários métodos:

| Método | O que faz |
|--------|-----------|
| `dist.pdf(x, loc, scale)` | Função densidade de probabilidade em `x` |
| `dist.cdf(x, loc, scale)` | Função de distribuição acumulada |
| `dist.ppf(q, loc, scale)` | Quantil (inversa da CDF) |
| `dist.rvs(loc, scale, size)` | Amostras aleatórias |
| `dist.mean(loc, scale)` | Média da distribuição |

Para `norm.pdf(x, 0, tau)`: calcula `N(x; μ=0, σ=tau)` — a densidade da Normal com média 0 e desvio padrão `tau`.

Para `laplace.pdf(x, 0, b)`: calcula `Laplace(x; μ=0, b=b)` — note que `scale=b` não é o desvio padrão (σ = b√2 para Laplace).

### Múltiplos parâmetros com `zip` de tuples

```python
for tau, lw_val, alpha_val in [(0.5, 3, 1.0), (1.0, 2, 0.7), (2.0, 1.5, 0.5)]:
```

Iterar sobre uma lista de tuples e desempacotar diretamente. Cada iteração: `tau=0.5, lw_val=3, alpha_val=1.0`. Curvas mais largas (tau maior) têm linhas mais finas e mais transparentes — destaque visual para distribuições mais concentradas.

### Comparação das penalidades

```python
ax.plot(theta_vis, theta_vis**2, 'tab:blue', lw=2.5, label='$\\|\\theta\\|^2$ (L2)')
ax.plot(theta_vis, np.abs(theta_vis), 'tab:orange', lw=2.5, label='$\\|\\theta\\|_1$ (L1)')
ax.plot(theta_vis, (np.abs(theta_vis) > 0).astype(float), ...)
```

`(np.abs(theta_vis) > 0).astype(float)` → array booleano convertido para float (0.0 ou 1.0) — representa a pseudo-norma L0 (vale 1 para qualquer θ ≠ 0).

---
## 8. Cross-Validation Variants — `StratifiedKFold` e `LeaveOneOut`

### O que o código faz

```python
# K-Fold padrão
kf = KFold(n_splits=K, shuffle=False)
for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(indices)):
    cores_fold = np.array([COR_TREINO]*N_vis)
    cores_fold[val_idx] = COR_VAL
    ax.barh([fold_idx]*N_vis, [1]*N_vis, left=range(N_vis),
            color=cores_fold, edgecolor='white', height=0.6)

# Stratified K-Fold
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=42)
for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(np.zeros(N_vis), y_strat)):
    n_pos_val = np.sum(y_strat[val_idx] == 1)
    ax.text(N_vis + 0.5, fold_idx, f'{n_pos_val}/{len(val_idx)} pos.', ...)

# Leave-One-Out
s_loo = cross_val_score(modelo_cv, X_cv, y_cv, cv=LeaveOneOut(), ...)
```

### O método `.split()` — gerador de índices

```python
for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(X)):
```

`kf.split(X)` é um **gerador** que produz pares `(indices_treino, indices_validação)` para cada fold. `enumerate` adiciona o número do fold.

**`tr_idx` e `val_idx`** são arrays de índices inteiros — usados para fatiar os dados com fancy indexing: `X[tr_idx]`, `y[val_idx]`.

### `StratifiedKFold` — por que precisa de `y`?

```python
skf.split(np.zeros(N_vis), y_strat)  # X pode ser qualquer coisa — y é o que importa
```

O Stratified precisa de `y` para saber as classes e garantir que cada fold preserve as proporções. O `X` passado para `.split()` é usado apenas para determinar `N` — pode ser um array de zeros.

### `LeaveOneOut` com `cross_val_score`

```python
s_loo = cross_val_score(modelo_cv, X_cv, y_cv, cv=LeaveOneOut(),
                        scoring='neg_mean_squared_error')
```

`LeaveOneOut()` cria um objeto CV com `n_splits=N`. Para `N=80`, roda 80 fits do modelo — pode ser lento. Mas `cross_val_score` aceita qualquer objeto CV, incluindo `LeaveOneOut()`.

### Visualizando divisões com `ax.barh` colorido

```python
cores_fold = np.array([COR_TREINO]*N_vis)  # inicializa tudo como treino
cores_fold[val_idx] = COR_VAL             # marca o fold de validação
ax.barh([fold_idx]*N_vis, [1]*N_vis, left=range(N_vis),
        color=cores_fold, edgecolor='white', height=0.6)
```

**Trick para visualização de CV:**
- `[fold_idx]*N_vis` → array de valores iguais ao índice do fold (posição y)
- `[1]*N_vis` → largura 1 para cada célula (barras de largura 1)
- `left=range(N_vis)` → cada barra começa onde a anterior terminou (0, 1, 2, ...)
- `edgecolor='white'` → separação visual entre células

Resultado: um mosaico de células onde cada linha é um fold e cada coluna é uma amostra.

---
## 9. Boxplots com `patch_artist`

### O que o código faz

```python
bp = ax.boxplot(dados_plot, labels=metodos, patch_artist=True,
                medianprops=dict(color='black', lw=2))
cores_box = ['tab:red', 'tab:blue', 'tab:green', 'tab:purple']
for patch, cor in zip(bp['boxes'], cores_box):
    patch.set_facecolor(cor); patch.set_alpha(0.5)
```

### `ax.boxplot` — anatomia e parâmetros

```python
ax.boxplot(dados,              # lista de arrays — um por caixa
           labels=metodos,     # rótulos no eixo x
           patch_artist=True,  # habilita preenchimento das caixas
           medianprops=dict(color='black', lw=2))
```

**Anatomia de um boxplot:**
- Linha central: **mediana**
- Bordas da caixa: Q1 (25%) e Q3 (75%) — **IQR (interquartile range)**
- Whiskers: geralmente Q1 − 1.5×IQR a Q3 + 1.5×IQR
- Pontos: **outliers** além dos whiskers

**`patch_artist=True`** permite preencher as caixas com cores (sem isso, as caixas são transparentes).

### Personalizando cores

```python
bp = ax.boxplot(...)  # retorna dicionário de artistas
for patch, cor in zip(bp['boxes'], cores_box):
    patch.set_facecolor(cor)
    patch.set_alpha(0.5)
```

`ax.boxplot()` retorna um dicionário com listas de artistas:
- `bp['boxes']` → as caixas (Rectangle patches)
- `bp['medians']` → linhas medianas
- `bp['whiskers']` → linhas dos whiskers
- `bp['caps']` → caps nos extremos dos whiskers
- `bp['fliers']` → pontos outliers

Iterar sobre `bp['boxes']` com `zip` permite colorir cada caixa diferente.

### Anotando `std` no topo

```python
for i, (m, dados) in enumerate(resultados_cv.items()):
    std = np.std(dados)
    ax.text(i+1, ax.get_ylim()[1]*0.97, f'std={std:.1f}',
            ha='center', fontsize=9, color=cores_box[i], fontweight='bold')
```

`ax.get_ylim()` retorna `(y_min, y_max)` — `[1]` acessa o máximo. Multiplicar por 0.97 posiciona o texto logo abaixo do topo da figura.

`i+1` porque boxplots têm posições x começando em 1 (não 0) por padrão no Matplotlib.

---
## 10. Coarse-to-Fine Log-Grid — Implementação Prática

### O que o código faz

```python
# Coarse sweep — amplo
lambdas_coarse = np.logspace(-5, 5, 20)
scores_coarse  = []
for lam in lambdas_coarse:
    s = cross_val_score(Ridge(alpha=lam), X_tr_t, y_tr_t, cv=kf_tune,
                        scoring='neg_mean_squared_error')
    scores_coarse.append(-s.mean())

melhor_coarse = lambdas_coarse[np.argmin(scores_coarse)]

# Fine sweep — ao redor do melhor coarse
lambdas_fine = np.logspace(np.log10(melhor_coarse) - 1,
                           np.log10(melhor_coarse) + 1, 30)
```

### `np.argmin` — índice do mínimo

```python
melhor_coarse = lambdas_coarse[np.argmin(scores_coarse)]
```

`np.argmin(array)` retorna o **índice** do menor valor (não o valor em si). Usado para acessar o λ correspondente ao menor erro de CV.

### Fine sweep em torno do melhor coarse

```python
lambdas_fine = np.logspace(
    np.log10(melhor_coarse) - 1,   # uma ordem de magnitude abaixo
    np.log10(melhor_coarse) + 1,   # uma ordem de magnitude acima
    30                              # 30 pontos nesse intervalo
)
```

`np.log10(melhor_coarse)` converte o λ ótimo para escala logarítmica. Subtrair/somar 1 expande uma ordem de magnitude para cada lado. Isso é equivalente a explorar `[λ*/10, λ*×10]`.

**Por que duas fases?**
- Coarse (20 pontos, 10 ordens de magnitude): encontra a **região** de λ ótimo
- Fine (30 pontos, 2 ordens de magnitude): encontra o **valor preciso** dentro da região

Total: 50 avaliações com 5× melhor resolução na região importante.

### Diagnóstico de under/over-regularização

```python
for ax, lam, label in zip(axes, lambdas_diag, labels_diag):
    modelo_d = make_pipeline(PolynomialFeatures(8), Ridge(alpha=lam))
    cv_s = cross_val_score(modelo_d, X_n, y_n, cv=min(5, N), ...)
    modelo_d.fit(X_n, y_n)
    erros_tr.append(mean_squared_error(y_n, modelo_d.predict(X_n)))
    erros_val.append(-cv_s.mean())
```

**`cv=min(5, N)`**: quando N é muito pequeno, não dá para fazer 5 folds (cada fold teria menos de 1 amostra). `min(5, N)` garante que o número de folds não excede o número de amostras.

---

## Resumo das Ferramentas

| Ferramenta | Uso no notebook |
|-----------|----------------|
| `mpatches.Patch(color, label)` | Legenda customizada com cores |
| `mpatches.FancyBboxPatch(xy, w, h, boxstyle)` | Caixas com bordas arredondadas |
| `ax.contour(X, Y, Z, levels, cmap)` | Linhas de nível da função de custo |
| `ax.fill(x_arr, y_arr, alpha, color)` | Preenche polígono definido por pontos |
| `plt.Circle((cx, cy), r)` + `ax.add_patch` | Círculo geométrico |
| `ax.set_aspect('equal')` | Escala igual nos eixos (obrigatório para geométricos) |
| `ElasticNet(alpha, l1_ratio, max_iter)` | L1 + L2 combinados |
| `[Model(alpha=l).fit(X, y).coef_ for l in ...]` | Regularization Path vetorizado |
| `theta.copy()` | Snapshot dos parâmetros (evita referência) |
| `np.inf` | Infinito positivo (inicialização de mínimos) |
| `ax.semilogy(...)` | Eixo y logarítmico (loss que cai em ordens de magnitude) |
| `np.random.binomial(1, p, size)` | Máscaras Bernoulli para Dropout |
| `scipy.stats.norm.pdf(x, loc, scale)` | PDF da Normal |
| `scipy.stats.laplace.pdf(x, loc, scale)` | PDF da Laplace |
| `KFold`, `StratifiedKFold`, `LeaveOneOut` | Estratégias de CV |
| `cv_obj.split(X)` | Gerador de índices treino/validação |
| `ax.boxplot(data, patch_artist=True)` | Boxplot com caixas preenchidas |
| `np.argmin(array)` | Índice do menor valor |
| `np.log10(x)` | Logaritmo base 10 (para fine sweep) |

---

## Como Proceder ao Usar Estes Padrões

1. **Para qualquer geometria nos eixos** (círculo, polígono): use `ax.set_aspect('equal')` — sem isso, as formas ficam distorcidas.

2. **Para Regularization Paths:** colete `model.coef_` em uma lista e converta para `np.array` — ganha operações de álgebra vetorizadas.

3. **Para Early Stopping:** sempre use `theta.copy()` ao salvar o checkpoint — nunca uma referência ao array mutável.

4. **Para CV com datasets desbalanceados:** sempre `StratifiedKFold`, nunca `KFold` padrão.

5. **Para busca de hiperparâmetros:** sempre Log-Grid (`np.logspace`) em vez de Linear-Grid (`np.linspace`) para parâmetros de regularização.

6. **Para Coarse-to-Fine:** o fine sweep usa `np.log10(melhor) ± 1` para se manter em escala logarítmica ao definir os limites do grid.